# Video Action Classification

## 1️⃣ Setup & Mount Drive

In [1]:
# =============================
# 1. SETUP
# =============================

from google.colab import drive
drive.mount('/content/drive')

import os
import cv2
import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from tqdm import tqdm

Mounted at /content/drive


2️⃣ Download dataset

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"dinrajkdinesh","key":"3820967f07e7d9577c408d1c73d729a7"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
#!/bin/bash
!kaggle datasets download abdallahwagih/ucf101-videos

Dataset URL: https://www.kaggle.com/datasets/abdallahwagih/ucf101-videos
License(s): apache-2.0
 96% 396M/415M [00:00<00:00, 516MB/s]
100% 415M/415M [00:00<00:00, 576MB/s]


In [5]:
!unzip /content/ucf101-videos.zip

Archive:  /content/ucf101-videos.zip
  inflating: test.csv                
  inflating: test/v_CricketShot_g01_c01.avi  
  inflating: test/v_CricketShot_g01_c02.avi  
  inflating: test/v_CricketShot_g01_c03.avi  
  inflating: test/v_CricketShot_g01_c04.avi  
  inflating: test/v_CricketShot_g01_c05.avi  
  inflating: test/v_CricketShot_g01_c06.avi  
  inflating: test/v_CricketShot_g01_c07.avi  
  inflating: test/v_CricketShot_g02_c01.avi  
  inflating: test/v_CricketShot_g02_c02.avi  
  inflating: test/v_CricketShot_g02_c03.avi  
  inflating: test/v_CricketShot_g02_c04.avi  
  inflating: test/v_CricketShot_g02_c05.avi  
  inflating: test/v_CricketShot_g02_c06.avi  
  inflating: test/v_CricketShot_g02_c07.avi  
  inflating: test/v_CricketShot_g03_c01.avi  
  inflating: test/v_CricketShot_g03_c02.avi  
  inflating: test/v_CricketShot_g03_c03.avi  
  inflating: test/v_CricketShot_g03_c04.avi  
  inflating: test/v_CricketShot_g03_c05.avi  
  inflating: test/v_CricketShot_g03_c06.avi  
  inf

In [27]:
import os
import pandas as pd


DATASET_ROOT = "/content/UCF101"

IMG_SIZE = 224
SEQUENCE_LENGTH = 16
BATCH_SIZE = 4
EPOCHS = 20

train_df = pd.read_csv(os.path.join(DATASET_ROOT, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATASET_ROOT, "test.csv"))

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))
print("Columns:", train_df.columns)

Train samples: 594
Test samples: 224
Columns: Index(['video_name', 'tag'], dtype='object')


4️⃣ Select Subset of Classes

In [29]:
ALL_CLASSES = sorted(train_df['tag'].unique())
SELECTED_CLASSES = ALL_CLASSES[:15]   # increase later if stable

print("Using classes:", SELECTED_CLASSES)

train_df = train_df[train_df['tag'].isin(SELECTED_CLASSES)]
test_df  = test_df[test_df['tag'].isin(SELECTED_CLASSES)]

Using classes: ['CricketShot', 'PlayingCello', 'Punch', 'ShavingBeard', 'TennisSwing']


Label Encoding

In [30]:
label_to_index = {label: idx for idx, label in enumerate(SELECTED_CLASSES)}
index_to_label = {idx: label for label, idx in label_to_index.items()}
NUM_CLASSES = len(SELECTED_CLASSES)

Build Path Lists

In [32]:
train_paths = []
train_labels = []

for _, row in train_df.iterrows():
    video_path = os.path.join(DATASET_ROOT, "train", row['video_name'])
    train_paths.append(video_path)
    train_labels.append(label_to_index[row['tag']])

val_paths = []
val_labels = []

for _, row in test_df.iterrows():
    video_path = os.path.join(DATASET_ROOT, "test", row['video_name'])
    val_paths.append(video_path)
    val_labels.append(label_to_index[row['tag']])

train_paths = np.array(train_paths)
train_labels = np.array(train_labels)
val_paths = np.array(val_paths)
val_labels = np.array(val_labels)

print("Train videos:", len(train_paths))
print("Validation videos:", len(val_paths))

Train videos: 594
Validation videos: 224


5️⃣ Frame Sampling (Uniform + Jitter)

In [33]:
def sample_frames_uniform_jitter(video_path, sequence_length=16):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < sequence_length:
        cap.release()
        return None

    segment_size = total_frames / sequence_length
    frames = []

    for i in range(sequence_length):
        start = int(i * segment_size)
        end = int((i + 1) * segment_size)
        idx = random.randint(start, max(start, end - 1))

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            cap.release()
            return None

        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return np.array(frames)

Memory-Safe Generator

In [34]:
def video_generator(paths, labels, batch_size=4, training=True):
    while True:
        indices = np.arange(len(paths))
        if training:
            np.random.shuffle(indices)

        for start in range(0, len(paths), batch_size):
            batch_indices = indices[start:start+batch_size]

            batch_videos = []
            batch_labels = []

            for idx in batch_indices:
                frames = sample_frames_uniform_jitter(paths[idx], SEQUENCE_LENGTH)
                if frames is None:
                    continue

                frames = frames / 255.0
                batch_videos.append(frames)
                batch_labels.append(labels[idx])

            if len(batch_videos) > 0:
                yield np.array(batch_videos), np.array(batch_labels)

Create Generators

In [35]:
train_gen = video_generator(train_paths, train_labels, BATCH_SIZE, training=True)
val_gen = video_generator(val_paths, val_labels, BATCH_SIZE, training=False)

steps_per_epoch = len(train_paths) // BATCH_SIZE
val_steps = len(val_paths) // BATCH_SIZE

8️⃣ Build End-to-End Model (TimeDistributed CNN + GRU)

In [37]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(SEQUENCE_LENGTH, IMG_SIZE, IMG_SIZE, 3)),
    layers.TimeDistributed(base_model),
    layers.TimeDistributed(layers.GlobalAveragePooling2D()),
    layers.GRU(256),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(), # Removed label_smoothing
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_4              │ (None, 16, 7, 7, 1280) │     2,257,984 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ (None, 16, 1280)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 256)            │     1,181,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,472,709 (13.25 MB)

 Trainable params: 1,214,725 (4.63 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

9️⃣ Callbacks

In [38]:
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=3),
    ModelCheckpoint("best_video_model.keras", save_best_only=True)
]

🔟 Train

In [39]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/20
148/148 ━━━━━━━━━━━━━━━━━━━━ 236s 1s/step - accuracy: 0.4068 - loss: 1.5219 - val_accuracy: 0.8839 - val_loss: 0.5940 - learning_rate: 1.0000e-04
Epoch 2/20
148/148 ━━━━━━━━━━━━━━━━━━━━ 153s 1s/step - accuracy: 0.8742 - loss: 0.4598 - val_accuracy: 0.8661 - val_loss: 0.3509 - learning_rate: 1.0000e-04
Epoch 3/20
148/148 ━━━━━━━━━━━━━━━━━━━━ 200s 1s/step - accuracy: 0.9449 - loss: 0.2132 - val_accuracy: 0.8795 - val_loss: 0.3077 - learning_rate: 1.0000e-04
Epoch 4/20
148/148 ━━━━━━━━━━━━━━━━━━━━ 152s 1s/step - accuracy: 0.9638 - loss: 0.1394 - val_accuracy: 0.9062 - val_loss: 0.2223 - learning_rate: 1.0000e-04
Epoch 5/20
148/148 ━━━━━━━━━━━━━━━━━━━━ 150s 1s/step - accuracy: 0.9899 - loss: 0.0716 - val_accuracy: 0.9062 - val_loss: 0.1989 - learning_rate: 1.0000e-04
Epoch 6/20
148/148 ━━━━━━━━━━━━━━━━━━━━ 149s 1s/step - accuracy: 0.9861 - loss: 0.0604 - val_accuracy: 0.9018 - val_loss: 0.2292 - learning_rate: 1.0000e-04
Epoch 7/20
148/148 ━━━━━━━━━━━━━━━━━━━━ 151s 1s/step - acc

1️⃣1️⃣ Save Deployment Model

In [40]:
model.save("final_video_action_model.keras")

In [42]:
model.evaluate(val_gen, steps=val_steps)

56/56 ━━━━━━━━━━━━━━━━━━━━ 44s 791ms/step - accuracy: 0.8864 - loss: 0.2787


[0.13900884985923767, 0.9330357313156128]

1️⃣4️⃣ Inference Function

In [46]:
def predict_video(video_path, model):
    frames = sample_frames_uniform_jitter(video_path, SEQUENCE_LENGTH)
    frames = frames / 255.0
    frames = np.expand_dims(frames, axis=0)

    prediction = model.predict(frames)
    class_idx = np.argmax(prediction)
    confidence = float(np.max(prediction))

    return index_to_label[class_idx], confidence

Inference Pipeline Test

In [47]:
predict_video("/content/UCF101/test/v_CricketShot_g06_c05.avi", model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step


('CricketShot', 0.9990620017051697)

## using a sample video of cricket from outside the dataset

In [48]:
predict_video("/content/cricket_sample.mp4", model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step


('CricketShot', 0.9984257221221924)

## using a sample video of ShavingBeard from outside the dataset

In [49]:
predict_video("/content/ShavingBeard_sample.mp4", model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step


('ShavingBeard', 0.899143397808075)